In [1]:
import pandas as pd
import numpy as np
import re

try:
    from unidecode import unidecode
except ModuleNotFoundError:
    !pip install unidecode
    from unidecode import unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 3.0 MB/s eta 0:00:00


In [2]:
import pandas as pd
from google.colab import files

# Upload:
# 1. processed_all_codes_icd10_and_flagged_icd9.csv
# 2. leaderboard_data.csv
uploaded = files.upload()

print("Uploaded files:")
for name in uploaded.keys():
    print(name)

combined_file = None
test_file = None

for name in uploaded.keys():
    lower = name.lower()

    if "processed_all_codes_icd10_and_flagged_icd9" in lower or "flagged_icd9" in lower:
        combined_file = name

    elif "leaderboard" in lower or "test" in lower:
        test_file = name

print("Detected combined ICD file:", combined_file)
print("Detected leaderboard/test file:", test_file)

if combined_file is None:
    raise ValueError("Could not detect processed_all_codes_icd10_and_flagged_icd9.csv. Please upload it.")

if test_file is None:
    raise ValueError("Could not detect leaderboard_data.csv. Please upload it.")

all_codes_df = pd.read_csv(combined_file)
test_df = pd.read_csv(test_file)

print("all_codes_df shape:", all_codes_df.shape)
print("test_df shape:", test_df.shape)

print("all_codes_df columns:")
print(all_codes_df.columns.tolist())

print("test_df columns:")
print(test_df.columns.tolist())

display(all_codes_df.head())
display(test_df.head())

Saving leaderboard_data (1).csv to leaderboard_data (1).csv
Saving processed_all_codes_icd10_and_flagged_icd9 (1).csv to processed_all_codes_icd10_and_flagged_icd9 (1).csv
Uploaded files:
leaderboard_data (1).csv
processed_all_codes_icd10_and_flagged_icd9 (1).csv
Detected combined ICD file: processed_all_codes_icd10_and_flagged_icd9 (1).csv
Detected leaderboard/test file: leaderboard_data (1).csv
all_codes_df shape: (13700, 7)
test_df shape: (6667, 2)
all_codes_df columns:
['Code', 'Literal', 'Code_clean', 'Literal_basic', 'in_icd_reference', 'D_P', 'Description']
test_df columns:
['id', 'Literal']


,Code,Literal,Code_clean,Literal_basic,in_icd_reference,D_P,Description
0,J9809,Hiperreactividad bronquial,J9809,Hiperreactividad bronquial,True,D,"Otras enfermedades de los bronquios, no clasif..."
1,J9801,broncoespástica,J9801,broncoespástica,True,D,Broncoespasmo agudo
2,I420,miocardiopatía dilatada,I420,miocardiopatía dilatada,True,D,Miocardiopatía dilatada
3,Y831,HTA irc 6,Y831,HTA irc 6,True,D,Cirugía con implante de dispositivo interno ar...
4,R5600,Crisis febriles atípicas,R5600,Crisis febriles atípicas,True,D,Convulsiones febriles simples


,id,Literal
0,1,AMNIODRENAJE
1,2,Hiperparatiroidismo primario
2,3,MIGRANYA parto
3,4,VHC
4,5,Absceso mama izq


In [4]:
def normalize_for_matching(text):
    text = str(text).lower()
    text = unidecode(text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-z0-9\s+/.-]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

if "Literal_match" not in test_df.columns:
    test_df["Literal_match"] = test_df["Literal"].apply(normalize_for_matching)

In [7]:
import re
from unidecode import unidecode

# ------------------------------------------------------------
# Text normalisation function
# ------------------------------------------------------------
def normalize_for_matching(text):
    text = str(text).lower()
    text = unidecode(text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-z0-9\s+/.-]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# ------------------------------------------------------------
# Create Literal_match if missing
# ------------------------------------------------------------
if "Literal_match" not in all_codes_df.columns:
    print("Creating Literal_match for all_codes_df...")
    all_codes_df["Literal_match"] = (
        all_codes_df["Literal"]
        .fillna("")
        .astype(str)
        .apply(normalize_for_matching)
    )

if "Literal_match" not in test_df.columns:
    print("Creating Literal_match for test_df...")
    test_df["Literal_match"] = (
        test_df["Literal"]
        .fillna("")
        .astype(str)
        .apply(normalize_for_matching)
    )

print("Done.")
print(all_codes_df.columns.tolist())
print(test_df.columns.tolist())

Creating Literal_match for all_codes_df...
Done.
['Code', 'Literal', 'Code_clean', 'Literal_basic', 'in_icd_reference', 'D_P', 'Description', 'Literal_match']
['id', 'Literal', 'Literal_match']


In [10]:
icd10_rows = all_codes_df[all_codes_df["in_icd_reference"] == True].copy()
icd9_rows = all_codes_df[all_codes_df["in_icd_reference"] == False].copy()

icd10_literals = set(icd10_rows["Literal_match"].dropna().astype(str))
icd9_literals = set(icd9_rows["Literal_match"].dropna().astype(str))
test_literals = set(test_df["Literal_match"].dropna().astype(str))

test_in_icd9_only = test_literals & (icd9_literals - icd10_literals)

test_icd9_only_df = test_df[test_df["Literal_match"].isin(test_in_icd9_only)].copy()
icd9_overlap_rows = icd9_rows[icd9_rows["Literal_match"].isin(test_in_icd9_only)].copy()

print("Test literals found only in ICD-9:", len(test_in_icd9_only))
print(test_icd9_only_df.shape)
print(icd9_overlap_rows.shape)



Test literals found only in ICD-9: 356
(478, 3)
(482, 8)


In [9]:
import os
from google.colab import files

os.makedirs("outputs", exist_ok=True)

test_icd9_only_df.to_csv("outputs/test_literals_found_only_in_flagged_icd9.csv", index=False)
icd9_overlap_rows.to_csv("outputs/flagged_icd9_rows_matching_test.csv", index=False)

files.download("outputs/test_literals_found_only_in_flagged_icd9.csv")
files.download("outputs/flagged_icd9_rows_matching_test.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
import pandas as pd
from google.colab import files

# Upload:
# 2018_I9gem.txt
uploaded = files.upload()

print("Uploaded files:")
for name in uploaded.keys():
    print(name)

gem_file = None

for name in uploaded.keys():
    lower = name.lower()
    if "i9gem" in lower and lower.endswith(".txt"):
        gem_file = name

print("Detected GEM file:", gem_file)

if gem_file is None:
    raise ValueError("Could not detect 2018_I9gem.txt. Please upload the official GEM txt file.")

Saving 2018_I9gem.txt to 2018_I9gem.txt
Uploaded files:
2018_I9gem.txt
Detected GEM file: 2018_I9gem.txt


In [12]:
def clean_code(code):
    return str(code).upper().strip().replace(".", "").replace(" ", "")

def load_gem_txt(path):
    rows = []
    with open(path, "r", encoding="latin1") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                rows.append({
                    "icd9_code_raw": parts[0],
                    "icd10_code_raw": parts[1],
                    "flags": parts[2] if len(parts) >= 3 else ""
                })
    return pd.DataFrame(rows)

gem_df = load_gem_txt(gem_file)

gem_df["icd9_code"] = gem_df["icd9_code_raw"].apply(clean_code)
gem_df["icd10_code"] = gem_df["icd10_code_raw"].apply(clean_code)
gem_df["mapped_category"] = gem_df["icd10_code"].str[0]

print("GEM shape:", gem_df.shape)
display(gem_df.head())

GEM shape: (24860, 6)


,icd9_code_raw,icd10_code_raw,flags,icd9_code,icd10_code,mapped_category
0,0010,A000,00000,0010,A000,A
1,0011,A001,00000,0011,A001,A
2,0019,A009,00000,0019,A009,A
3,0020,A0100,10000,0020,A0100,A
4,0021,A011,00000,0021,A011,A


In [13]:
icd9_overlap_rows["icd9_code"] = icd9_overlap_rows["Code_clean"].apply(clean_code)

icd9_mapped = icd9_overlap_rows.merge(
    gem_df[["icd9_code", "icd10_code", "mapped_category", "flags"]],
    on="icd9_code",
    how="left"
)

literal_cat_counts = (
    icd9_mapped
    .dropna(subset=["mapped_category"])
    .groupby(["Literal_match", "mapped_category"])
    .size()
    .reset_index(name="count")
)

literal_totals = (
    literal_cat_counts
    .groupby("Literal_match")["count"]
    .sum()
    .reset_index(name="total_count")
)

literal_cat_counts = literal_cat_counts.merge(literal_totals, on="Literal_match", how="left")
literal_cat_counts["purity"] = literal_cat_counts["count"] / literal_cat_counts["total_count"]

safe_literal_map = (
    literal_cat_counts
    .sort_values(["Literal_match", "count"], ascending=[True, False])
    .drop_duplicates("Literal_match")
)

safe_literal_map = safe_literal_map[safe_literal_map["purity"] == 1.0].copy()

official_gem_literal_to_cat = dict(zip(
    safe_literal_map["Literal_match"],
    safe_literal_map["mapped_category"]
))

print("Safe GEM literal mappings:", len(official_gem_literal_to_cat))

Safe GEM literal mappings: 281


In [15]:
import os
from google.colab import files

os.makedirs("outputs", exist_ok=True)

safe_literal_map.to_csv("outputs/safe_gem_literal_map.csv", index=False)

files.download("outputs/safe_gem_literal_map.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
#submission disagreement analysis
# ============================================================
# Submission Disagreement Analysis
# Compare different submissions row by row
# ============================================================

import pandas as pd
import numpy as np
from google.colab import files

# Upload your submission files, for example:
# submission_selective_context.csv
# submission_category.csv
# submission_icd10_svc_with_official_gem_override.csv
# submission_icd10_plus_official_gem_icd9_weight_05.csv
uploaded = files.upload()

subs = {}

for name in uploaded.keys():
    if name.endswith(".csv"):
        df = pd.read_csv(name)
        print(name, df.shape, df.columns.tolist())

        if "y_category" in df.columns:
            subs[name] = df
        else:
            print("Skipping because no y_category column:", name)

merged = None

for name, df in subs.items():
    pred_col = (
        name
        .replace(".csv", "")
        .replace(" ", "_")
        .replace("(", "")
        .replace(")", "")
    )

    if "Literal" in df.columns:
        temp = df[["id", "Literal", "y_category"]].copy()
    else:
        temp = df[["id", "y_category"]].copy()

    temp = temp.rename(columns={"y_category": pred_col})

    if merged is None:
        merged = temp
    else:
        keep_cols = ["id", pred_col]
        merged = merged.merge(temp[keep_cols], on="id", how="inner")

pred_cols = [c for c in merged.columns if c not in ["id", "Literal"]]

print("Merged shape:", merged.shape)
print("Prediction columns:")
print(pred_cols)

display(merged.head())

Saving submission_selective_context_safe_gem_override.csv to submission_selective_context_safe_gem_override.csv
Saving submission_icd10_all_svc_selective_context.csv to submission_icd10_all_svc_selective_context.csv
Saving submission_icd10_all_svc_only.csv to submission_icd10_all_svc_only.csv
submission_selective_context_safe_gem_override.csv (6667, 3) ['id', 'Literal', 'y_category']
submission_icd10_all_svc_selective_context.csv (6667, 3) ['id', 'Literal', 'y_category']
submission_icd10_all_svc_only.csv (6667, 3) ['id', 'Literal', 'y_category']
Merged shape: (6667, 5)
Prediction columns:
['submission_selective_context_safe_gem_override', 'submission_icd10_all_svc_selective_context', 'submission_icd10_all_svc_only']


,id,Literal,submission_selective_context_safe_gem_override,submission_icd10_all_svc_selective_context,submission_icd10_all_svc_only
0,1,AMNIODRENAJE,1,1,1
1,2,Hiperparatiroidismo primario,E,E,E
2,3,MIGRANYA parto,O,O,O
3,4,VHC,B,B,B
4,5,Absceso mama izq,N,N,N


In [17]:
# ============================================================
# Use selective context as reference baseline
# ============================================================

selective_col = [c for c in pred_cols if "selective_context" in c][0]

print("Using selective context baseline:", selective_col)

change_summary = []

for col in pred_cols:
    if col == selective_col:
        continue

    changed = merged[col] != merged[selective_col]

    change_summary.append({
        "method": col,
        "changed_vs_selective_context": int(changed.sum()),
        "changed_rate": changed.mean()
    })

change_summary_df = pd.DataFrame(change_summary)

display(
    change_summary_df.sort_values(
        "changed_vs_selective_context",
        ascending=False
    )
)

Using selective context baseline: submission_selective_context_safe_gem_override


,method,changed_vs_selective_context,changed_rate
0,submission_icd10_all_svc_selective_context,53,0.00795
1,submission_icd10_all_svc_only,53,0.00795


In [18]:
#inspect changed rows
# ============================================================
# Inspect rows changed by each method
# ============================================================

for col in pred_cols:
    if col == selective_col:
        continue

    changed_rows = merged[merged[col] != merged[selective_col]].copy()

    print("\n" + "="*90)
    print("Method:", col)
    print("Changed rows vs selective context:", len(changed_rows))

    print("\nSelective context predictions on changed rows:")
    print(changed_rows[selective_col].value_counts().head(25))

    print("\nNew method predictions on changed rows:")
    print(changed_rows[col].value_counts().head(25))

    display(changed_rows[["id", "Literal", selective_col, col]].head(100))


Method: submission_icd10_all_svc_selective_context
Changed rows vs selective context: 53

Selective context predictions on changed rows:
submission_selective_context_safe_gem_override
K    11
I    10
N     8
J     7
E     6
C     4
Q     2
D     2
R     1
B     1
G     1
Name: count, dtype: int64

New method predictions on changed rows:
submission_icd10_all_svc_selective_context
0    17
Z     8
Q     5
O     4
K     3
N     2
B     2
R     2
1     2
G     2
I     1
H     1
M     1
T     1
C     1
8     1
Name: count, dtype: int64


,id,Literal,submission_selective_context_safe_gem_override,submission_icd10_all_svc_selective_context
6,7,cos estrany,N,0
22,23,HPB,N,O
165,166,desgarro de 1r grado,Q,O
380,381,biopsia de médula ósea,I,0
435,436,SIADH,E,Z
457,458,absceso retrofaríngeo,J,N
524,525,historia iam,I,Z
731,732,biopsia selectiva ganglio centinela,I,0
768,769,aplasia,D,Q
1012,1013,historia IAM,I,Z



Method: submission_icd10_all_svc_only
Changed rows vs selective context: 53

Selective context predictions on changed rows:
submission_selective_context_safe_gem_override
K    11
I    10
N     8
J     7
E     6
C     4
Q     2
D     2
R     1
B     1
G     1
Name: count, dtype: int64

New method predictions on changed rows:
submission_icd10_all_svc_only
0    17
Z     8
Q     5
O     4
K     3
N     2
B     2
R     2
1     2
G     2
I     1
H     1
M     1
T     1
C     1
8     1
Name: count, dtype: int64


,id,Literal,submission_selective_context_safe_gem_override,submission_icd10_all_svc_only
6,7,cos estrany,N,0
22,23,HPB,N,O
165,166,desgarro de 1r grado,Q,O
380,381,biopsia de médula ósea,I,0
435,436,SIADH,E,Z
457,458,absceso retrofaríngeo,J,N
524,525,historia iam,I,Z
731,732,biopsia selectiva ganglio centinela,I,0
768,769,aplasia,D,Q
1012,1013,historia IAM,I,Z


In [19]:
# ============================================================
# Save disagreement analysis
# ============================================================

merged["n_unique_preds"] = merged[pred_cols].nunique(axis=1)

disagree_rows = merged[merged["n_unique_preds"] > 1].copy()

print("Rows where at least one model disagrees:", len(disagree_rows))

display(disagree_rows.head(100))

disagree_rows.to_csv("submission_disagreement_rows.csv", index=False)
change_summary_df.to_csv("submission_change_summary.csv", index=False)

files.download("submission_disagreement_rows.csv")
files.download("submission_change_summary.csv")

Rows where at least one model disagrees: 53


,id,Literal,submission_selective_context_safe_gem_override,submission_icd10_all_svc_selective_context,submission_icd10_all_svc_only,n_unique_preds
6,7,cos estrany,N,0,0,2
22,23,HPB,N,O,O,2
165,166,desgarro de 1r grado,Q,O,O,2
380,381,biopsia de médula ósea,I,0,0,2
435,436,SIADH,E,Z,Z,2
457,458,absceso retrofaríngeo,J,N,N,2
524,525,historia iam,I,Z,Z,2
731,732,biopsia selectiva ganglio centinela,I,0,0,2
768,769,aplasia,D,Q,Q,2
1012,1013,historia IAM,I,Z,Z,2


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>